## **Mejoras Implementadas en el Proceso de Limpieza**

### **Problemas Identificados y Soluciones**

**🔍 Problemas Detectados en Archivos Originales:**
- **Puntos superfluos**: Líneas que comenzaban con `. ` 
- **Fracciones mal separadas**: Números romanos y arábigos separados del contenido
- **Espaciado inconsistente**: Espacios múltiples y formato irregular
- **Encabezados repetidos**: Headers administrativos duplicados
- **Líneas fragmentadas**: Párrafos divididos incorrectamente

**✅ Soluciones Implementadas:**

1. **`_fix_leading_dots()`** - Elimina puntos al inicio de líneas que causaban formato incorrecto
2. **Detección mejorada de fracciones** - Patrones específicos para números romanos, arábigos y alfabéticos
3. **`_final_cleanup()`** - Limpieza integral de líneas vacías y espacios múltiples
4. **Mejor reconocimiento de párrafos** - Identificación precisa de inicios de párrafo legal
5. **Normalización de Unicode** - Manejo de diferentes tipos de espacios y caracteres especiales

**📊 Resultados:**
- **Formato consistente**: Una línea por párrafo legal
- **Estructura preservada**: Artículos, fracciones y títulos claramente delimitados  
- **Texto limpio**: Sin artefactos de extracción PDF
- **Reducción de tamaño**: Eliminación eficiente de contenido redundante

**🎯 Casos Específicos Corregidos:**
- `I. II. III.` → Fracciones correctamente separadas como párrafos individuales
- `Artículo 1.1.-` → Formato consistente de artículos
- Eliminación de headers: `Secretaría de Asuntos Parlamentarios`, `Última reforma`, etc.
- Espaciado normalizado en todo el documento

## **Required Dependencies & Imports**

In [2]:
# Core Python libraries
from __future__ import annotations          # Enable forward type references
import re                                   # Regular expressions for text pattern matching
import statistics                           # Statistical calculations for text analysis

# Data structures and type hints
from dataclasses import dataclass, field    # Structured data classes for law metadata
from pathlib import Path                    # Cross-platform file system path handling
from typing import Dict, List, Optional, Tuple, Union, Set, Any  # Type annotations

# External libraries
#%pip install PyMuPDF pandas unidecode
import fitz                                 # PyMuPDF - PDF text extraction library
import pandas as pd                         # Data manipulation and CSV handling
from unidecode import unidecode             # Unicode normalization and accent removal

## **Directory Structure & Data Flow Configuration**

### **Input Directories**
- **`Raw/`** - Source PDF files containing legal documents
- **`index.csv`** - Catalog mapping file numbers to law metadata

### **Processing Pipeline Directories**
- **`temp/raw_txt/`** - Step 1: Raw text extracted from PDFs  
- **`temp/clean/`** - Step 2: Cleaned and normalized text

### **Output Directories**  
- **`errores/`** - Error logs and validation reports

This configuration ensures a clear separation of processing stages and enables easy debugging and quality control.

In [3]:
# ============== Directory Configuration ==============

# Base working directory (current notebook location)
BASE_DIR     = Path.cwd()

# === INPUT PATHS ===
RAW_DIR      = BASE_DIR / "Raw"                 # Source PDF files location
CATALOG_CSV  = RAW_DIR / "index.csv"           # Metadata catalog for PDFs

# === OUTPUT ROOT ===
OUTPUT_DIR   = BASE_DIR / "Refined"             # All processed outputs go here
TEMP_DIR     = BASE_DIR / "temp"                # Temporary processing files

# === INTERMEDIATE PROCESSING DIRECTORIES ===
RAW_TXT_DIR  = OUTPUT_DIR / TEMP_DIR / "raw_txt"   # Step 1: Raw PDF text extraction
CLEAN_DIR    = OUTPUT_DIR / TEMP_DIR / "clean"     # Step 2: Cleaned text files

# === DOCUMENT TYPE SEPARATION (Step 3 outputs) ===
LEY_DIR      = OUTPUT_DIR / "leyes"             # Main law content
DECR_DIR     = OUTPUT_DIR / "decretos"          # Government decree sections  
TRANS_DIR    = OUTPUT_DIR / "transitorios"      # Transitional provisions

# === FINAL OUTPUTS ===
JSON_DIR     = OUTPUT_DIR / "json"              # Structured JSON documents
ERRORES_DIR  = OUTPUT_DIR / "errores"           # Error logs and validation reports

# Create all necessary directories (parents=True creates nested paths)
for d in [OUTPUT_DIR, RAW_TXT_DIR, TEMP_DIR, CLEAN_DIR, LEY_DIR, DECR_DIR, TRANS_DIR, JSON_DIR, ERRORES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Directory structure created successfully!")
print(f"Base directory: {BASE_DIR}")
print(f"Input PDFs: {RAW_DIR}")  
print(f"Catalog: {CATALOG_CSV}")
print(f"Final JSON output: {JSON_DIR}")

Directory structure created successfully!
Base directory: c:\Users\braul\Documents\_ITAMLaptop\Datalab\DataMakers\Leyes\15
Input PDFs: c:\Users\braul\Documents\_ITAMLaptop\Datalab\DataMakers\Leyes\15\Raw
Catalog: c:\Users\braul\Documents\_ITAMLaptop\Datalab\DataMakers\Leyes\15\Raw\index.csv
Final JSON output: c:\Users\braul\Documents\_ITAMLaptop\Datalab\DataMakers\Leyes\15\Refined\json


## **Utility Functions & Text Processing Tools**

### **Core Functionality**
- **Text Normalization** - Clean and standardize text for consistent processing
- **File I/O Operations** - Safe file writing with UTF-8 encoding
- **Error Logging** - Structured error reporting with contextual information  
- **Statistical Analysis** - Count lines, words, and characters for quality metrics
- **String Sanitization** - Create safe filenames and normalize accented characters

These utilities ensure robust text processing and provide comprehensive error tracking throughout the pipeline.

In [4]:
# ============== Text Processing Utilities ==============

def slugify(s: str) -> str:
    """
    Convert text to URL-safe slug format.
    - Removes accents using unidecode
    - Converts to lowercase
    - Replaces non-alphanumeric chars with underscores]
    - Collapses multiple underscores to single ones
    """
    s = unidecode(s).lower()                    # Remove accents, convert to lowercase
    s = re.sub(r"[^a-z0-9]+", "_", s)          # Replace non-alphanumeric with underscores
    return re.sub(r"_+", "_", s).strip("_") or "x"  # Clean up multiple underscores

def norm_lower(s: str) -> str:
    """Normalize text: remove accents, lowercase, collapse whitespace."""
    return re.sub(r"\s+", " ", unidecode(s).lower().strip())

def caps_line(s: str) -> str:
    """Convert text to uppercase and remove accents for header matching."""
    return unidecode(s).upper().strip()

def write_text(path: Path, content: str) -> None:
    """Safely write text content to file with UTF-8 encoding."""
    path.write_text(content, encoding="utf-8")

def write_error(base: str, kind: str, message: str, extra: Dict | None = None) -> None:
    """
    Log structured error information to JSON file.
    
    Args:
        base: File identifier (e.g., '0001')
        kind: Error category (e.g., 'catalog_missing', 'parse_error')
        message: Human-readable error description
        extra: Additional context data
    """
    rec = {"file": base, "kind": kind, "message": message}
    if extra:
        rec.update(extra)
    
    # Create safe filename for error log
    error_filename = f"{slugify(base)}_{slugify(kind)}.json"
    (ERRORES_DIR / error_filename).write_text(
        json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8"
    )

def count_stats(text: str) -> Dict[str, int]:
    """
    Calculate text statistics for quality metrics.
    
    Returns:
        Dictionary with 'lines', 'words', and 'chars' counts
    """
    return {
        "lines": text.count("\n") + (1 if text else 0),  # Count newlines + 1
        "words": len(re.findall(r"\S+", text)),           # Count non-whitespace sequences
        "chars": len(text)                                # Total character count
    }

def _norm_caps(s: str) -> str:
    """
    Internal helper: normalize text to uppercase alphanumeric with spaces.
    Used for header pattern matching.
    """
    t = unidecode(s).upper()                    # Remove accents, uppercase
    t = re.sub(r"[^A-Z0-9]+", " ", t)          # Keep only letters/numbers
    return re.sub(r"\s+", " ", t).strip()       # Collapse whitespace

In [5]:
# ============== Law Metadata & Catalog Management ==============

@dataclass(frozen=True)
class LawMeta:
    """
    Immutable metadata container for legal documents.
    
    Attributes:
        num_est: State number identifier
        file_num: Zero-padded file number (e.g., '0001')
        law_name: Full name of the law
        link: Source URL or reference link
        first_two_caps: Auto-generated uppercase version of first two words
                       (used for document structure detection)
    """
    num_est: str
    file_num: str  
    law_name: str
    link: str
    first_two_caps: str = field(init=False)  # Computed automatically

    def __post_init__(self):
        """
        Automatically extract and normalize the first two words of law name.
        This is used for identifying the law title within document text.
        """
        # Tokenize and normalize the law name
        toks = [t for t in norm_lower(self.law_name).split() if t]
        first_two = " ".join(toks[:2]) if toks else ""
        
        # Set the computed field (frozen dataclass requires object.__setattr__)
        object.__setattr__(self, "first_two_caps", first_two.upper())

def load_catalog(path: Path) -> Dict[str, LawMeta]:
    """
    Load law metadata from CSV catalog file.
    
    Args:
        path: Path to catalog CSV file
        
    Returns:
        Dictionary mapping file_num to LawMeta objects
        
    Raises:
        ValueError: If required columns are missing from CSV
    """
    # Read CSV with string dtype to preserve leading zeros
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    df.columns = [c.lower() for c in df.columns]  # Normalize column names
    
    # Verify required columns exist
    req = {"num_est", "file_num", "law_name", "link"}
    miss = req - set(df.columns)
    if miss:
        raise ValueError(f"CSV missing required columns: {miss}")
    
    # Build catalog dictionary
    out: Dict[str, LawMeta] = {}
    for _, r in df.iterrows():
        meta = LawMeta(
            num_est=(r["num_est"] or "").strip(),
            file_num=(r["file_num"] or "").strip().zfill(4),  # Ensure 4-digit padding
            law_name=(r["law_name"] or "").strip(),
            link=(r["link"] or "").strip(),
        )
        out[meta.file_num] = meta
    
    return out

## **Step 1: PDF Text Extraction**

### **Purpose**
Extract raw text content from PDF legal documents using PyMuPDF (fitz). This step converts binary PDF files into plain text while preserving layout and structure as much as possible.

### **Process**
1. **PDF Reading** - Open each PDF file in the Raw directory
2. **Page-by-Page Extraction** - Extract text from each page sequentially  
3. **Text Concatenation** - Combine all pages with double newlines as separators
4. **File Output** - Save raw text to `temp/raw_txt/` directory
5. **Statistics Tracking** - Record file processing metrics and errors

### **Quality Control**
- Validates files against catalog metadata
- Tracks processing statistics (lines, words, characters)
- Logs missing files and extraction errors
- Generates manifest for downstream processing

In [6]:
def read_pdf(pdf_path: Path) -> str:
    """
    Extract text content from a PDF file using PyMuPDF.
    
    Args:
        pdf_path: Path to the PDF file to process
        
    Returns:
        Concatenated text content from all pages
        
    Note:
        - Processes pages sequentially to maintain document order
        - Adds double newlines between pages for section separation
        - Handles multi-page documents automatically
    """
    with fitz.open(pdf_path) as pdf_file:
        text_content = ""
        
        # Process each page in order
        for page_num in range(len(pdf_file)):
            page = pdf_file[page_num]
            text = page.get_text()                    # Extract plain text
            text_content += text + "\n\n"             # Add page separator

    return text_content

In [7]:
def step1_extract_raw(catalog_csv: Path = CATALOG_CSV) -> pd.DataFrame:
    """
    Step 1: Extract raw text from all PDF files in the Raw directory.
    
    Process:
    1. Load catalog metadata for file validation
    2. Find all PDF files in Raw directory  
    3. Extract text from each PDF using PyMuPDF
    4. Save raw text files to temp/raw_txt/ directory
    5. Generate processing statistics and manifest
    
    Args:
        catalog_csv: Path to catalog file for metadata validation
        
    Returns:
        DataFrame with processing statistics for each file
    """
    print("Starting Step 1: PDF Text Extraction...")
    
    # Load catalog for file validation and metadata
    catalog = load_catalog(catalog_csv)
    pdfs = sorted(RAW_DIR.glob("*.pdf"))
    
    print(f"Found {len(pdfs)} PDF files to process")
    print(f"Catalog contains {len(catalog)} entries")

    all_recs: List[Dict] = []
    
    for pdf_path in pdfs:
        # Extract base filename (e.g., '0001.pdf' -> '0001')
        base = pdf_path.stem.zfill(4) 
        
        # Validate against catalog
        if base not in catalog:
            write_error(base, "catalog_missing", 
                       "file_num not found in catalog", 
                       {"pdf": pdf_path.name})
            print(f"WARNING: {pdf_path.name} not found in catalog - skipping")
            continue

        # Extract text from PDF
        print(f"Processing {pdf_path.name}...")
        raw_layout = read_pdf(pdf_path)

        # Save raw text output
        raw_out = RAW_TXT_DIR / f"raw_{base}.txt"
        write_text(raw_out, raw_layout)

        # Record processing statistics
        rec_raw = {
            "stage": "raw",
            "source_pdf": pdf_path.name,
            "base": base
        }
        rec_raw.update(count_stats(raw_layout))  # Add line/word/char counts
        all_recs.append(rec_raw)

    # Create processing manifest
    df = pd.DataFrame(all_recs)
    if not df.empty:
        manifest_path = OUTPUT_DIR / "manifest_raw.csv"
        df.to_csv(manifest_path, index=False, encoding="utf-8")
        print(f"Manifest saved to {manifest_path}")
    
    print(f"Step 1 Complete: {len(df)} raw txt files written to {RAW_TXT_DIR}")
    return df

# === EXECUTE STEP 1 ===
print("=" * 60)
df_raw = step1_extract_raw(CATALOG_CSV)
print("=" * 60)

# Display processing summary
if not df_raw.empty:
    print("\n**Raw Text Extraction Summary:**")
    print(f"   Total files processed: {len(df_raw)}")
    print(f"   Average lines per file: {df_raw['lines'].mean():.1f}")
    print(f"   Average words per file: {df_raw['words'].mean():.1f}")
    print(f"   Total characters extracted: {df_raw['chars'].sum():,}")
    
df_raw.head()

Starting Step 1: PDF Text Extraction...
Found 235 PDF files to process
Catalog contains 235 entries
Processing 0001.pdf...
Processing 0002.pdf...
Processing 0003.pdf...
Processing 0004.pdf...
Processing 0005.pdf...
Processing 0006.pdf...
Processing 0007.pdf...
Processing 0008.pdf...
Processing 0009.pdf...
Processing 0010.pdf...
Processing 0011.pdf...
Processing 0012.pdf...
Processing 0013.pdf...
Processing 0014.pdf...
Processing 0015.pdf...
Processing 0016.pdf...
Processing 0017.pdf...
Processing 0018.pdf...
Processing 0019.pdf...
Processing 0020.pdf...
Processing 0021.pdf...
Processing 0022.pdf...
Processing 0023.pdf...
Processing 0024.pdf...
Processing 0025.pdf...
Processing 0026.pdf...
Processing 0027.pdf...
Processing 0028.pdf...
Processing 0029.pdf...
Processing 0030.pdf...
Processing 0031.pdf...
Processing 0032.pdf...
Processing 0033.pdf...
Processing 0034.pdf...
Processing 0035.pdf...
Processing 0036.pdf...
Processing 0037.pdf...
Processing 0038.pdf...
Processing 0039.pdf...
Pro

,stage,source_pdf,base,lines,words,chars
0,raw,0001.pdf,0001,3711,45209,284029
1,raw,0002.pdf,0002,13662,136272,906535
2,raw,0003.pdf,0003,12143,116477,732794
3,raw,0004.pdf,0004,2316,24282,155330
4,raw,0005.pdf,0005,6486,63160,402099


## **Step 2: Text Cleaning & Normalization**

### **Purpose**
Clean and normalize raw text extracted from PDFs to prepare for structural analysis. This step removes PDF artifacts, standardizes formatting, and improves text quality for downstream processing.

### **Cleaning Operations**
- **Line Break Normalization** - Convert Windows/Mac line endings to Unix format
- **Whitespace Consolidation** - Collapse multiple spaces and excessive line breaks
- **Paragraph Preservation** - Maintain paragraph structure while removing artifacts
- **Title Matching** - Remove duplicate title lines that match catalog metadata

### **Quality Improvements**
- Removes PDF extraction artifacts and formatting inconsistencies
- Preserves document structure and readability
- Standardizes text encoding and character representation
- Prepares text for accurate structural parsing in Step 3

In [15]:
# ============== Step 2 — Text Cleaning & Normalization (One-Paragraph-One-Line) ==============
# Objetivo:
# - Cada párrafo queda en UNA sola línea.
# - Preservar la separabilidad de encabezados (TÍTULO/CAPÍTULO/SECCIÓN/ART./TRANSITORIOS/DECRETO/NÚMERO)
#   y fracciones (romanas, numéricas, alfabéticas), tratándolos como inicios de nuevo párrafo.
# - Eliminar espacios redundantes, saltos de línea extra y artefactos comunes de extracción PDF.
# - Estandarizar Unicode (NFC, NBSP→espacio, quitar cero-width), dehyphenation segura.
# - Idempotente y listo para Step 3 (parsing estructural).
#
# Integra con utilidades de tu proyecto si existen (fallbacks seguros incluidos).

from __future__ import annotations
import re
import unicodedata
from pathlib import Path
from collections import Counter
from typing import Dict, List, Optional
import sys

# ----------------------- Integraciones del proyecto (con fallbacks) -----------------------
try:
    RAW_TXT_DIR
except NameError:
    RAW_TXT_DIR = Path("temp/raw_txt")

try:
    CLEAN_DIR
except NameError:
    CLEAN_DIR = Path("temp/clean")

try:
    OUTPUT_DIR
except NameError:
    OUTPUT_DIR = Path("Refined")

try:
    CATALOG_CSV
except NameError:
    CATALOG_CSV = Path("catalog.csv")

def _fallback_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def _fallback_write_error(base, code, msg, extra=None) -> None:
    sys.stderr.write(f"[clean][{base}] {code}: {msg} | extra={extra}\n")

def _fallback_count_stats(text: str) -> Dict[str, int]:
    return {
        "lines": text.count("\n") + (0 if text.endswith("\n") else 1 if text else 0),
        "words": len(text.split()),
        "chars": len(text),
    }

write_text   = globals().get("write_text", _fallback_write_text)
write_error  = globals().get("write_error", _fallback_write_error)
count_stats  = globals().get("count_stats", _fallback_count_stats)
load_catalog = globals().get("load_catalog", None)  # opcional

# ---------------------------- Patrones estructurales clave ----------------------------
HEAD_RE = re.compile(
    r'^\s*(T[ÍI]TULO|CAP[ÍI]TULO|SECCI[ÓO]N|ART[ÍI]CULO|DECRETO|TRANSITORIOS?|N[ÚU]MERO)\b',
    re.IGNORECASE
)
# "Art." o "Art " como atajo de ARTÍCULO
ARTICLE_RE = re.compile(r'^\s*Art(?:[íi]culo|\.?)\b', re.IGNORECASE)

# Enumeraciones: romanas, numéricas, alfabéticas (a), b), etc.)
ENUM_RE = re.compile(
    r'^\s*((?:I{1,4}|V?I{0,3}|X{1,3})\s*(?:\.|-)|\d+\s*(?:\.|\)|-)|[A-Za-zÁÉÍÓÚÜÑ]\))'
)

# Contadores/pies de página típicos
PAGE_RE_1 = re.compile(r'^\s*[-–—]?\s*\d+\s*[-–—]?\s*$')
PAGE_RE_2 = re.compile(r'^\s*P[aá]gina\s+\d+(\s+de\s+\d+)?\s*$', re.IGNORECASE)

# Variante espaciada de TRANSITORIOS
TRANS_SPACED_RE = re.compile(r'^\s*T\s*R\s*A\s*N\s*S\s*I\s*T\s*O\s*R\s*I\s*O\s*S\s*$', re.IGNORECASE)

# ------------------------------ Utilidades de limpieza ------------------------------
def _normalize_unicode(s: str) -> str:
    s = s.replace('\r\n', '\n').replace('\r', '\n')
    s = s.replace('\ufeff', '')     # BOM
    s = s.replace('\u00A0', ' ')    # NBSP -> espacio
    s = re.sub('[\u200b-\u200d]', '', s)  # zero-width
    s = unicodedata.normalize('NFC', s)
    return s

def _strip_trailing_spaces(s: str) -> str:
    return "\n".join(ln.rstrip() for ln in s.split("\n"))

def _remove_page_counters_and_repeated_headers(s: str) -> str:
    # 1) Quitar folios/páginas
    lines = [ln for ln in s.split("\n") if not (PAGE_RE_1.match(ln) or PAGE_RE_2.match(ln))]
    # 2) Quitar encabezados/pies repetidos idénticos (>=3 veces) que no sean estructurales
    counts = Counter(l for l in lines if l.strip() and not HEAD_RE.match(l) and not ENUM_RE.match(l))
    repeated = {l for l, c in counts.items() if c >= 3}
    return "\n".join(l for l in lines if l not in repeated)

def _normalize_transitorios_spaced(s: str) -> str:
    out = []
    for ln in s.split("\n"):
        out.append("TRANSITORIOS" if TRANS_SPACED_RE.match(ln.strip()) else ln)
    return "\n".join(out)

def _is_paragraph_starter(ln: str) -> bool:
    if not ln.strip():
        return True
    return bool(HEAD_RE.match(ln) or ARTICLE_RE.match(ln) or ENUM_RE.match(ln))

def _join_hard_wrap(prev: str, cur: str) -> str:
    # Une respetando dehyphenation segura y un solo espacio
    if prev.endswith('-') and cur and cur[:1].islower():
        return prev[:-1] + cur
    return (prev + ' ' + cur).replace('  ', ' ')

def _paragraphs_to_one_line(s: str) -> str:
    """
    Recorre el texto línea por línea y construye párrafos de UNA línea.
    Empieza un nuevo párrafo cuando:
      - la línea está en blanco, o
      - coincide con HEAD/ARTICLE/ENUM (encabezados y fracciones).
    En caso contrario, concatena con la línea previa dentro del mismo párrafo.
    """
    out: List[str] = []
    acc: Optional[str] = None

    for raw_ln in s.split("\n"):
        ln = raw_ln.strip()
        if not ln:  # línea en blanco -> finaliza párrafo actual
            if acc is not None and acc.strip():
                out.append(acc.strip())
                acc = None
            continue

        if _is_paragraph_starter(ln):
            # Cierra el párrafo previo y empieza uno nuevo
            if acc is not None and acc.strip():
                out.append(acc.strip())
            acc = ln
        else:
            # Continúa el párrafo actual
            if acc is None:
                acc = ln
            else:
                acc = _join_hard_wrap(acc, ln)

    if acc is not None and acc.strip():
        out.append(acc.strip())

    # Sin líneas en blanco: una línea por párrafo.
    return "\n".join(out)

def _intraline_space_normalize(s: str) -> str:
    # Colapsa espacios internos y asegura separación mínima tras tokens legales comunes
    s = re.sub(r'[ \t]{2,}', ' ', s)
    s = re.sub(r'(^|\n)(Artículo\s+\d+(?:[º°]\.?|\.))\s*', r'\1\2 ', s, flags=re.IGNORECASE)
    s = re.sub(r'(^|\n)(Art\.\s*\d+\.?)\s*', r'\1\2 ', s, flags=re.IGNORECASE)
    s = re.sub(r'(^|\n)((?:I{1,4}|V?I{0,3}|X{1,3})\s*(?:\.|-))\s*', r'\1\2 ', s)
    s = re.sub(r'(^|\n)(\d+\s*(?:\.|\)|-))\s*', r'\1\2 ', s)
    s = re.sub(r'(^|\n)([A-Za-zÁÉÍÓÚÜÑ]\))\s*', r'\1\2 ', s)
    return s

# --------------------------------- API principal ---------------------------------
def clean_raw_text(raw: str, title_candidate: Optional[str] = None) -> str:
    """
    Limpieza conservadora con "una línea por párrafo".
    Pasos:
      1) Unicode & controles (NFC, NBSP→espacio, quitar zero-width, normalizar CR/CRLF)
      2) Quitar folios/pies de página y encabezados/pies repetidos (>=3)
      3) Normalizar variantes espaciadas de TRANSITORIOS
      4) Reflujo: construir párrafos de UNA línea (HEAD/ART./ENUM inician párrafo)
      5) Normalización intralínea y tidy final
    """
    txt = _normalize_unicode(raw)
    txt = _strip_trailing_spaces(txt)
    txt = _remove_page_counters_and_repeated_headers(txt)
    txt = _normalize_transitorios_spaced(txt)

    # *** clave: una línea por párrafo ***
    txt = _paragraphs_to_one_line(txt)

    # Ajustes finos de espacios
    txt = _intraline_space_normalize(txt)

    # Tidy final: idempotencia y salto final único
    txt = txt.strip() + "\n"
    return txt

# ------------------------------------ Driver ------------------------------------
def step2_clean_raw(catalog_csv: Path = CATALOG_CSV):
    """
    Step 2: Limpia y normaliza archivos raw -> clean_{base}.txt
    - Lee RAW_TXT_DIR / raw_*.txt
    - Escribe CLEAN_DIR / clean_{base}.txt
    - Manifiesto con estadísticas (si pandas disponible)
    - Logs claros por archivo
    """
    def log(msg: str, level: str = "INFO"):
        print(f"[{level}] {msg}", flush=True)

    print("Starting Step 2: Text Cleaning & Normalization (one paragraph = one line)...")
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)

    catalog = None
    if load_catalog:
        try:
            catalog = load_catalog(catalog_csv)
            log(f"Catalog loaded: {catalog_csv}")
        except Exception as e:
            write_error("global", "catalog_load_failed", f"Could not load catalog: {e}", {"catalog_csv": str(catalog_csv)})
            log(f"Catalog load failed: {e}", level="WARN")

    raw_files = sorted(RAW_TXT_DIR.glob("raw_*.txt"))
    log(f"Found {len(raw_files)} raw text files in {RAW_TXT_DIR}")

    records: List[Dict] = []
    total_chars_raw = 0
    total_chars_clean = 0
    cleaned_count = 0
    skipped_count = 0
    failed_count = 0

    for raw_path in raw_files:
        base = raw_path.stem.replace("raw_", "")
        fname = raw_path.name

        if catalog is not None and base not in catalog:
            write_error(base, "catalog_missing", "file_num not found in catalog (clean stage)", {"raw_file": fname})
            log(f"{fname} → skipping (not in catalog)", level="WARN")
            skipped_count += 1
            continue

        try:
            log(f"Cleaning {fname} …")
            raw_text = raw_path.read_text(encoding="utf-8")
            meta = catalog[base] if catalog is not None else None
            title = getattr(meta, "law_name", None) if meta is not None else None

            cleaned = clean_raw_text(raw_text, title)

            out_path = CLEAN_DIR / f"clean_{base}.txt"
            write_text(out_path, cleaned)

            stats_raw = count_stats(raw_text)
            stats_clean = count_stats(cleaned)
            total_chars_raw += stats_raw.get("chars", 0)
            total_chars_clean += stats_clean.get("chars", 0)

            rec = {"stage": "clean", "source_raw": fname, "base": base, **stats_clean}
            records.append(rec)
            cleaned_count += 1

            log(f"Wrote {out_path.name} (lines={stats_clean.get('lines')}, words={stats_clean.get('words')}, chars={stats_clean.get('chars')})")

        except Exception as e:
            write_error(base, "clean_failed", f"Exception during cleaning: {e}", {"raw_file": fname})
            log(f"{fname} → cleaning failed: {e}", level="ERROR")
            failed_count += 1

    # Manifest opcional
    try:
        import pandas as pd
        if records:
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            import pandas as pd
            df = pd.DataFrame(records)
            manifest_path = OUTPUT_DIR / "manifest_clean.csv"
            df.to_csv(manifest_path, index=False, encoding="utf-8")
            log(f"Manifest saved to {manifest_path}")
        else:
            df = pd.DataFrame()
    except Exception as e:
        log(f"Could not write manifest (pandas missing or error: {e})", level="WARN")
        try:
            import pandas as pd
            df = pd.DataFrame(records) if records else pd.DataFrame()
        except Exception:
            df = None

    # Resumen
    reduction = (1 - (total_chars_clean / total_chars_raw)) * 100 if total_chars_raw else 0.0
    log(f"Summary: cleaned={cleaned_count}, skipped={skipped_count}, failed={failed_count}")
    if total_chars_raw:
        log(f"Text size reduction: {reduction:.1f}%")
    log(f"Step 2 complete: {cleaned_count} file(s) written to {CLEAN_DIR}")

    return df if (df is not None and not df.empty) else None

# ------------------------------- Ejecución directa --------------------------------
if __name__ == "__main__":
    _ = step2_clean_raw(CATALOG_CSV)


Starting Step 2: Text Cleaning & Normalization (one paragraph = one line)...
[INFO] Found 235 raw text files in temp\raw_txt
[INFO] Cleaning raw_0001.txt …
[INFO] Wrote clean_0001.txt (lines=678, words=44619, chars=275725)
[INFO] Cleaning raw_0002.txt …
[INFO] Wrote clean_0002.txt (lines=4021, words=135703, chars=880016)
[INFO] Cleaning raw_0003.txt …
[INFO] Wrote clean_0003.txt (lines=3701, words=118129, chars=722367)
[INFO] Cleaning raw_0004.txt …
[INFO] Wrote clean_0004.txt (lines=694, words=24192, chars=150210)
[INFO] Cleaning raw_0005.txt …
[INFO] Wrote clean_0005.txt (lines=1828, words=64013, chars=396930)
[INFO] Cleaning raw_0006.txt …
[INFO] Wrote clean_0006.txt (lines=2061, words=85865, chars=551809)
[INFO] Cleaning raw_0007.txt …
[INFO] Wrote clean_0007.txt (lines=5908, words=176017, chars=1081778)
[INFO] Cleaning raw_0008.txt …
[INFO] Wrote clean_0008.txt (lines=2204, words=104094, chars=687037)
[INFO] Cleaning raw_0009.txt …
[INFO] Wrote clean_0009.txt (lines=1493, words=66